In [ ]:
!pip install anthropic -q

import anthropic
import base64
import json
import os
import re
import pandas as pd
import numpy as np
from PIL import Image
import io
from tqdm import tqdm
from google.colab import drive, userdata

drive.mount('/content/drive')

BASE              = 'YOUR_DRIVE_PATH'
HUMAN_FOLDER      = f'{BASE}/images_human_fixed'
IMAGE_FOLDER      = f'{BASE}/images'
RESULT_FOLDER     = f'{BASE}/results'
HAND_FEATURE_PATH = f'{BASE}/labels/vqa_hand_features.json'
MODEL_FOLDER_HAND = f'{BASE}/models_hybrid_hand'

os.makedirs(MODEL_FOLDER_HAND, exist_ok=True)

from google.colab import userdata
client = anthropic.Anthropic(
    api_key=userdata.get('ANTHROPIC_API_KEY')
)

test_df  = pd.read_csv(f'{BASE}/labels/human_test.csv')
train_df = pd.read_csv(f'{BASE}/labels/human_train.csv')
val_df   = pd.read_csv(f'{BASE}/labels/human_val.csv')

for d in [test_df, train_df, val_df]:
    d['dementia_risk'] = d['dementia_risk'].astype(bool)

def encode_image(img_path):
    try:
        img    = Image.open(img_path).convert('RGB')
        buffer = io.BytesIO()
        img.save(buffer, format='JPEG', quality=95)
        return base64.standard_b64encode(
            buffer.getvalue()
        ).decode('utf-8')
    except:
        return None

print(f'Test : {len(test_df)} รูป')

In [ ]:
HAND_VQA_PROMPT = """
You are analyzing a Clock Drawing Test (CDT) image.
Focus ONLY on the clock hands.

The correct time to draw is 11:10
(hour hand near 11, minute hand near 2)

Return ONLY a JSON object:
{
  "has_hour_hand": <"Yes"/"No">,
  "has_minute_hand": <"Yes"/"No">,
  "hand_count": <int 0-3>,
  "hour_hand_points_to": <int 1-12 or -1 if absent>,
  "minute_hand_points_to": <int 1-12 or -1 if absent>,
  "hour_hand_correct": <"Yes"/"No"/"Absent">,
  "minute_hand_correct": <"Yes"/"No"/"Absent">,
  "both_hands_correct": <"Yes"/"No">,
  "hour_hand_length": <"short"/"medium"/"long"/"absent">,
  "minute_hand_length": <"short"/"medium"/"long"/"absent">,
  "length_ratio_correct": <"Yes"/"No"/"Unclear">,
  "hands_overlap": <"Yes"/"No">,
  "hand_clarity": <int 1-5>,
  "time_shown": <"string e.g. 11:10 or unknown">
}

Note:
- Hour hand should point toward 11
- Minute hand should point toward 2 (10 minutes)
- Hour hand should be shorter than minute hand
"""

def extract_hand_features(img_path):
    """ดึง hand-specific features"""
    img_b64 = encode_image(img_path)
    if img_b64 is None:
        return None

    try:
        response = client.messages.create(
            model='claude-haiku-4-5',
            max_tokens=400,
            messages=[{
                'role': 'user',
                'content': [
                    {
                        'type': 'image',
                        'source': {
                            'type'      : 'base64',
                            'media_type': 'image/jpeg',
                            'data'      : img_b64,
                        }
                    },
                    {
                        'type': 'text',
                        'text': HAND_VQA_PROMPT
                    }
                ]
            }]
        )

        text = response.content[0].text.strip()

        import re
        json_match = re.search(
            r'\{.*\}', text, re.DOTALL
        )
        if json_match:
            text = json_match.group()
        else:
            text = text.replace(
                '```json', ''
            ).replace('```', '').strip()

        return json.loads(text)

    except json.JSONDecodeError:
        return None
    except Exception as e:
        print(f'Error: {e}')
        return None

# print('''
#   has_hour_hand        ← มีเข็มชั่วโมงไหม
#   has_minute_hand      ← มีเข็มนาทีไหม
#   hour_hand_points_to  ← ชี้ไปเลข 11 ไหม
#   minute_hand_points_to← ชี้ไปเลข 2 ไหม
#   both_hands_correct   ← ถูกทั้งคู่ไหม
#   hour_hand_length     ← สั้น/กลาง/ยาว
#   minute_hand_length   ← สั้น/กลาง/ยาว
#   length_ratio_correct ← ขนาดต่างกันถูกไหม
#   time_shown           ← เวลาที่วาด
# ''')

In [ ]:
sample_files = test_df['filename'].tolist()[:5]
for filename in sample_files:
    src = f'{HUMAN_FOLDER}/{filename}'
    if not os.path.exists(src):
        src = f'{IMAGE_FOLDER}/{filename.replace(".jpg",".tif")}'

    print(f'รูป: {filename}')
    features = extract_hand_features(src)

    if features:
        print(f'  has_hour_hand      : {features.get("has_hour_hand")}')
        print(f'  has_minute_hand    : {features.get("has_minute_hand")}')
        print(f'  hour → points to  : {features.get("hour_hand_points_to")}')
        print(f'  minute → points to: {features.get("minute_hand_points_to")}')
        print(f'  both_correct       : {features.get("both_hands_correct")}')
        print(f'  length_ratio_ok    : {features.get("length_ratio_correct")}')
        print(f'  time_shown         : {features.get("time_shown")}')
    else:
        print('  ดึงไม่ได้')
    print()

In [ ]:
all_df = pd.read_csv(
    f'{BASE}/labels/human_labels.csv'
)

if os.path.exists(HAND_FEATURE_PATH):
    with open(HAND_FEATURE_PATH) as f:
        hand_features = json.load(f)
    print(f'มี progress เดิม: {len(hand_features)}')
else:
    hand_features = {}

remaining = [
    row for _, row in all_df.iterrows()
    if row['filename'] not in hand_features
    or 'error' in hand_features.get(
        row['filename'], {}
    )
]

print(f'ยังต้อง extract: {len(remaining)} รูป')
print(f'ประมาณ: {len(remaining)*2/60:.1f} นาที')

error_count = 0

for row in tqdm(remaining):
    filename = row['filename']

    src_human = f'{HUMAN_FOLDER}/{filename}'
    src_orig  = f'{IMAGE_FOLDER}/{filename.replace(".jpg",".tif")}'

    if os.path.exists(src_human):
        src_path = src_human
    elif os.path.exists(src_orig):
        src_path = src_orig
    else:
        error_count += 1
        continue

    features = extract_hand_features(src_path)

    if features:
        hand_features[filename] = features
    else:
        error_count += 1
        hand_features[filename] = {'error': True}

    if len(hand_features) % 50 == 0:
        with open(HAND_FEATURE_PATH, 'w') as f:
            json.dump(hand_features, f)

with open(HAND_FEATURE_PATH, 'w') as f:
    json.dump(hand_features, f, indent=2)

success = sum(
    1 for v in hand_features.values()
    if 'error' not in v
)
print(f'\nExtract เสร็จ')
print(f'  สำเร็จ: {success} รูป')
print(f'  Error : {error_count} รูป')

In [ ]:
def hand_features_to_vector(features):
    """Hand-specific feature vector"""
    if features is None or 'error' in features:
        return np.zeros(10)

    vec = []

    vec.append(
        1.0 if features.get(
            'has_hour_hand'
        ) == 'Yes' else 0.0
    )

    vec.append(
        1.0 if features.get(
            'has_minute_hand'
        ) == 'Yes' else 0.0
    )

    vec.append(
        features.get('hand_count', 0) / 3.0
    )

    h_correct = features.get('hour_hand_correct')
    vec.append(
        1.0 if h_correct == 'Yes' else
        0.5 if h_correct == 'Absent' else 0.0
    )

    m_correct = features.get('minute_hand_correct')
    vec.append(
        1.0 if m_correct == 'Yes' else
        0.5 if m_correct == 'Absent' else 0.0
    )

    vec.append(
        1.0 if features.get(
            'both_hands_correct'
        ) == 'Yes' else 0.0
    )

    lr = features.get('length_ratio_correct')
    vec.append(
        1.0 if lr == 'Yes' else
        0.5 if lr == 'Unclear' else 0.0
    )

    h_pos = features.get('hour_hand_points_to', -1)
    vec.append(
        h_pos / 12.0 if h_pos > 0 else 0.0
    )

    m_pos = features.get('minute_hand_points_to', -1)
    vec.append(
        m_pos / 12.0 if m_pos > 0 else 0.0
    )

    vec.append(
        features.get('hand_clarity', 1) / 5.0
    )

    return np.array(vec, dtype=np.float32)

sample = list(hand_features.values())[0]
vec    = hand_features_to_vector(sample)
print(f'Hand feature vector shape: {vec.shape}')
print(f'Vector: {vec}')

In [ ]:
import json

vqa_path = f'{BASE}/labels/vqa_features.json'
with open(vqa_path) as f:
    vqa_features = json.load(f)

hand_path = f'{BASE}/labels/vqa_hand_features.json'
with open(hand_path) as f:
    hand_features = json.load(f)

vqa_success  = sum(1 for v in vqa_features.values()
                   if 'error' not in v)
hand_success = sum(1 for v in hand_features.values()
                   if 'error' not in v)

print(f'VQA features : {vqa_success}/{len(vqa_features)} รูป')
print(f'Hand features: {hand_success}/{len(hand_features)} รูป')

In [ ]:
import numpy as np

def features_to_vector(features):
    if features is None or 'error' in features:
        return np.zeros(12)
    vec = []
    vec.append(features.get('digit_count', 0) / 12.0)
    digits = features.get('digits_present', [])
    vec.append(len(digits) / 12.0)
    vec.append(1.0 if features.get(
        'digit_positions_correct') == 'Yes' else 0.0)
    vec.append(
        features.get('quadrant_balance', 0) / 4.0)
    wq = features.get('worst_quadrant', 'none')
    wq_map = {
        'top-left'    : [1,0,0,0,0],
        'top-right'   : [0,1,0,0,0],
        'bottom-left' : [0,0,1,0,0],
        'bottom-right': [0,0,0,1,0],
        'none'        : [0,0,0,0,1],
    }
    vec.extend(wq_map.get(wq, [0,0,0,0,1]))
    vec.append(1.0 if features.get(
        'has_hands') == 'Yes' else 0.0)
    vec.append(features.get('hand_count', 0) / 3.0)
    vec.append(1.0 if features.get(
        'hands_at_correct_position') == 'Yes' else 0.0)
    vec.append(
        features.get('spatial_regularity', 1) / 5.0)
    vec.append(
        features.get('overall_quality', 1) / 5.0)
    return np.array(vec, dtype=np.float32)

def hand_features_to_vector(features):
    if features is None or 'error' in features:
        return np.zeros(10)
    vec = []
    vec.append(1.0 if features.get(
        'has_hour_hand') == 'Yes' else 0.0)
    vec.append(1.0 if features.get(
        'has_minute_hand') == 'Yes' else 0.0)
    vec.append(features.get('hand_count', 0) / 3.0)
    h_correct = features.get('hour_hand_correct')
    vec.append(
        1.0 if h_correct == 'Yes' else
        0.5 if h_correct == 'Absent' else 0.0)
    m_correct = features.get('minute_hand_correct')
    vec.append(
        1.0 if m_correct == 'Yes' else
        0.5 if m_correct == 'Absent' else 0.0)
    vec.append(1.0 if features.get(
        'both_hands_correct') == 'Yes' else 0.0)
    lr = features.get('length_ratio_correct')
    vec.append(
        1.0 if lr == 'Yes' else
        0.5 if lr == 'Unclear' else 0.0)
    h_pos = features.get('hour_hand_points_to', -1)
    vec.append(h_pos / 12.0 if h_pos > 0 else 0.0)
    m_pos = features.get('minute_hand_points_to', -1)
    vec.append(m_pos / 12.0 if m_pos > 0 else 0.0)
    vec.append(
        features.get('hand_clarity', 1) / 5.0)
    return np.array(vec, dtype=np.float32)

def combined_features_to_vector(
    filename, vqa_feats, hand_feats
):
    vqa_vec  = features_to_vector(
        vqa_feats.get(filename, {})
    )
    hand_vec = hand_features_to_vector(
        hand_feats.get(filename, {})
    )
    return np.concatenate([vqa_vec, hand_vec])

sample_combined = combined_features_to_vector(
    test_df['filename'].iloc[0],
    vqa_features, hand_features
)
COMBINED_DIM = len(sample_combined)
print(f'Combined dim: {COMBINED_DIM}')
print(f'  VQA : {len(features_to_vector({}))}')
print(f'  Hand: {len(hand_features_to_vector({}))}')

In [ ]:
def combined_features_to_vector(
    filename, vqa_feats, hand_feats
):
    """รวม VQA + Hand features"""
    vqa_vec  = features_to_vector(
        vqa_feats.get(filename, {})
    )
    hand_vec = hand_features_to_vector(
        hand_feats.get(filename, {})
    )
    return np.concatenate([vqa_vec, hand_vec])

sample_combined = combined_features_to_vector(
    test_df['filename'].iloc[0],
    vqa_features, hand_features
)
COMBINED_DIM = len(sample_combined)
print(f'Combined dim: {COMBINED_DIM}')
print(f'  VQA : {len(features_to_vector({}))}')
print(f'  Hand: {len(hand_features_to_vector({}))}')
print(f'  Total        : {COMBINED_DIM}')

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch

In [ ]:
class CDTCombinedDataset(Dataset):
    def __init__(self, df, image_folder,
                 vqa_feats, hand_feats,
                 aug_folder=None, transform=None):
        self.df           = df.reset_index(drop=True)
        self.image_folder = image_folder
        self.vqa_feats    = vqa_feats
        self.hand_feats   = hand_feats
        self.aug_folder   = aug_folder
        self.transform    = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        filename = row['filename']

        if (self.aug_folder and
                filename.startswith('aug')):
            img_path = f'{self.aug_folder}/{filename}'
        else:
            img_path = f'{self.image_folder}/{filename}'

        try:
            img = Image.open(img_path).convert('RGB')
        except:
            img = Image.new('RGB', (224, 224),
                           (255, 255, 255))

        if self.transform:
            img = self.transform(img)

        # Combined features
        feat = torch.tensor(
            combined_features_to_vector(
                filename,
                self.vqa_feats,
                self.hand_feats
            ),
            dtype=torch.float32
        )

        labels = torch.tensor([
            row['domain_A'], row['domain_B'],
            row['domain_C'], row['domain_D'],
            row['domain_E']
        ], dtype=torch.float32)

        return img, feat, labels

In [ ]:
device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)

In [ ]:
!pip install timm -q

import torch
import torch.nn as nn
import torch.optim as optim
import timm
import pandas as pd
import numpy as np
import json
import os
import random
from PIL import Image, ImageEnhance
from torchvision import transforms, models
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    confusion_matrix, roc_auc_score,
    f1_score, accuracy_score,
    mean_absolute_error
)
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import gaussian_kde
from google.colab import drive

drive.mount('/content/drive')

BASE              = 'YOUR_DRIVE_PATH'
HUMAN_FOLDER      = f'{BASE}/images_human_fixed'
IMAGE_FOLDER      = f'{BASE}/images'
AUG_FOLDER        = f'{BASE}/images_human_aug'
RESULT_FOLDER     = f'{BASE}/results'
FEATURE_PATH      = f'{BASE}/labels/vqa_features.json'
HAND_FEATURE_PATH = f'{BASE}/labels/vqa_hand_features.json'
MODEL_FOLDER_HAND = f'{BASE}/models_hybrid_hand'
MODEL_FOLDER_VQA  = f'{BASE}/models_hybrid'
MODEL_FOLDER_US   = f'{BASE}/models_human_undersample'

device = torch.device(
    'cuda' if torch.cuda.is_available() else 'cpu'
)
CUTOFF = 6

domain_names = {
    0: 'A: digit_count',
    1: 'B: worst_quadrant',
    2: 'C: spatial',
    3: 'D: hands_present',
    4: 'E: hands_placement',
}
colors_true = {
    0: '#3F51B5',
    1: '#00BCD4',
    2: '#8BC34A',
}

test_df  = pd.read_csv(f'{BASE}/labels/human_test.csv')
train_df = pd.read_csv(f'{BASE}/labels/human_train.csv')
val_df   = pd.read_csv(f'{BASE}/labels/human_val.csv')
for d in [test_df, train_df, val_df]:
    d['dementia_risk'] = d['dementia_risk'].astype(bool)

with open(FEATURE_PATH) as f:
    vqa_features = json.load(f)
with open(HAND_FEATURE_PATH) as f:
    hand_features = json.load(f)

def features_to_vector(features):
    if features is None or 'error' in features:
        return np.zeros(12)
    vec = []
    vec.append(features.get('digit_count', 0) / 12.0)
    digits = features.get('digits_present', [])
    vec.append(len(digits) / 12.0)
    vec.append(1.0 if features.get(
        'digit_positions_correct') == 'Yes' else 0.0)
    vec.append(
        features.get('quadrant_balance', 0) / 4.0)
    wq = features.get('worst_quadrant', 'none')
    wq_map = {
        'top-left'    : [1,0,0,0,0],
        'top-right'   : [0,1,0,0,0],
        'bottom-left' : [0,0,1,0,0],
        'bottom-right': [0,0,0,1,0],
        'none'        : [0,0,0,0,1],
    }
    vec.extend(wq_map.get(wq, [0,0,0,0,1]))
    vec.append(1.0 if features.get(
        'has_hands') == 'Yes' else 0.0)
    vec.append(features.get('hand_count', 0) / 3.0)
    vec.append(1.0 if features.get(
        'hands_at_correct_position') == 'Yes' else 0.0)
    vec.append(
        features.get('spatial_regularity', 1) / 5.0)
    vec.append(
        features.get('overall_quality', 1) / 5.0)
    return np.array(vec, dtype=np.float32)

def hand_features_to_vector(features):
    if features is None or 'error' in features:
        return np.zeros(10)
    vec = []
    vec.append(1.0 if features.get(
        'has_hour_hand') == 'Yes' else 0.0)
    vec.append(1.0 if features.get(
        'has_minute_hand') == 'Yes' else 0.0)
    vec.append(features.get('hand_count', 0) / 3.0)
    h = features.get('hour_hand_correct')
    vec.append(1.0 if h == 'Yes' else
               0.5 if h == 'Absent' else 0.0)
    m = features.get('minute_hand_correct')
    vec.append(1.0 if m == 'Yes' else
               0.5 if m == 'Absent' else 0.0)
    vec.append(1.0 if features.get(
        'both_hands_correct') == 'Yes' else 0.0)
    lr = features.get('length_ratio_correct')
    vec.append(1.0 if lr == 'Yes' else
               0.5 if lr == 'Unclear' else 0.0)
    h_pos = features.get('hour_hand_points_to', -1)
    vec.append(h_pos / 12.0 if h_pos > 0 else 0.0)
    m_pos = features.get('minute_hand_points_to', -1)
    vec.append(m_pos / 12.0 if m_pos > 0 else 0.0)
    vec.append(
        features.get('hand_clarity', 1) / 5.0)
    return np.array(vec, dtype=np.float32)

def combined_features_to_vector(
    filename, vqa_feats, hand_feats
):
    return np.concatenate([
        features_to_vector(vqa_feats.get(filename, {})),
        hand_features_to_vector(
            hand_feats.get(filename, {})
        )
    ])

VQA_DIM      = len(features_to_vector({}))
HAND_DIM     = len(hand_features_to_vector({}))
COMBINED_DIM = VQA_DIM + HAND_DIM

class CDTDataset(Dataset):
    def __init__(self, df, image_folder,
                 aug_folder=None, transform=None):
        self.df           = df.reset_index(drop=True)
        self.image_folder = image_folder
        self.aug_folder   = aug_folder
        self.transform    = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        filename = row['filename']
        if (self.aug_folder and
                filename.startswith('aug')):
            img_path = f'{self.aug_folder}/{filename}'
        else:
            img_path = f'{self.image_folder}/{filename}'
        try:
            img = Image.open(img_path).convert('RGB')
        except:
            img = Image.new('RGB', (224,224),(255,255,255))
        if self.transform:
            img = self.transform(img)
        labels = torch.tensor([
            row['domain_A'], row['domain_B'],
            row['domain_C'], row['domain_D'],
            row['domain_E']
        ], dtype=torch.float32)
        return img, labels

class CDTHybridDataset(Dataset):
    def __init__(self, df, image_folder,
                 vqa_feats, aug_folder=None,
                 transform=None):
        self.df           = df.reset_index(drop=True)
        self.image_folder = image_folder
        self.vqa_feats    = vqa_feats
        self.aug_folder   = aug_folder
        self.transform    = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        filename = row['filename']
        if (self.aug_folder and
                filename.startswith('aug')):
            img_path = f'{self.aug_folder}/{filename}'
        else:
            img_path = f'{self.image_folder}/{filename}'
        try:
            img = Image.open(img_path).convert('RGB')
        except:
            img = Image.new('RGB', (224,224),(255,255,255))
        if self.transform:
            img = self.transform(img)
        feat = torch.tensor(
            features_to_vector(
                self.vqa_feats.get(filename, {})
            ), dtype=torch.float32)
        labels = torch.tensor([
            row['domain_A'], row['domain_B'],
            row['domain_C'], row['domain_D'],
            row['domain_E']
        ], dtype=torch.float32)
        return img, feat, labels

class CDTCombinedDataset(Dataset):
    def __init__(self, df, image_folder,
                 vqa_feats, hand_feats,
                 aug_folder=None, transform=None):
        self.df           = df.reset_index(drop=True)
        self.image_folder = image_folder
        self.vqa_feats    = vqa_feats
        self.hand_feats   = hand_feats
        self.aug_folder   = aug_folder
        self.transform    = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        filename = row['filename']
        if (self.aug_folder and
                filename.startswith('aug')):
            img_path = f'{self.aug_folder}/{filename}'
        else:
            img_path = f'{self.image_folder}/{filename}'
        try:
            img = Image.open(img_path).convert('RGB')
        except:
            img = Image.new('RGB', (224,224),(255,255,255))
        if self.transform:
            img = self.transform(img)
        feat = torch.tensor(
            combined_features_to_vector(
                filename,
                self.vqa_feats,
                self.hand_feats
            ), dtype=torch.float32)
        labels = torch.tensor([
            row['domain_A'], row['domain_B'],
            row['domain_C'], row['domain_D'],
            row['domain_E']
        ], dtype=torch.float32)
        return img, feat, labels

class CDTModel(nn.Module):
    def __init__(self, backbone_name='resnet50',
                 num_domains=5, num_classes=3):
        super().__init__()
        if backbone_name == 'resnet50':
            backbone = models.resnet50(
                weights=models.ResNet50_Weights.IMAGENET1K_V1
            )
            self.feature_dim = 2048
            for param in backbone.parameters():
                param.requires_grad = False
            self.backbone = nn.Sequential(
                *list(backbone.children())[:-1]
            )
        elif backbone_name == 'vit':
            self.backbone = timm.create_model(
                'vit_base_patch16_224',
                pretrained=True, num_classes=0
            )
            self.feature_dim = 768
            for param in self.backbone.parameters():
                param.requires_grad = False
        self.backbone_name = backbone_name
        self.shared = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.feature_dim, 512),
            nn.ReLU(), nn.Dropout(0.3)
        )
        self.heads_cls = nn.ModuleList([
            nn.Linear(512, num_classes)
            for _ in range(num_domains)
        ])
        self.heads_reg = nn.ModuleList([
            nn.Linear(512, 1)
            for _ in range(num_domains)
        ])

    def forward(self, x, mode='reg'):
        if self.backbone_name == 'resnet50':
            feat = self.backbone(x)
        else:
            feat = self.backbone(x)
            feat = feat.unsqueeze(-1).unsqueeze(-1)
        feat = self.shared(feat)
        if mode == 'reg':
            return [head(feat).squeeze(-1)
                    for head in self.heads_reg]
        else:
            return [head(feat)
                    for head in self.heads_cls]

class CDTHybridModel(nn.Module):
    def __init__(self, backbone_name='vit',
                 vqa_dim=12, num_domains=5):
        super().__init__()
        if backbone_name == 'vit':
            self.backbone = timm.create_model(
                'vit_base_patch16_224',
                pretrained=True, num_classes=0
            )
            vis_dim = 768
        else:
            backbone = models.resnet50(
                weights=models.ResNet50_Weights.IMAGENET1K_V1
            )
            self.backbone = nn.Sequential(
                *list(backbone.children())[:-1]
            )
            vis_dim = 2048
        self.backbone_name = backbone_name
        for param in self.backbone.parameters():
            param.requires_grad = False
        self.vis_shared = nn.Sequential(
            nn.Flatten(),
            nn.Linear(vis_dim, 512),
            nn.ReLU(), nn.Dropout(0.3)
        )
        self.vqa_encoder = nn.Sequential(
            nn.Linear(vqa_dim, 64),
            nn.ReLU(), nn.Dropout(0.2),
            nn.Linear(64, 32), nn.ReLU()
        )
        self.fusion = nn.Sequential(
            nn.Linear(512+32, 256),
            nn.ReLU(), nn.Dropout(0.3)
        )
        self.heads_reg = nn.ModuleList([
            nn.Linear(256, 1)
            for _ in range(num_domains)
        ])
        self.heads_cls = nn.ModuleList([
            nn.Linear(256, 3)
            for _ in range(num_domains)
        ])

    def forward(self, img, vqa_feat, mode='reg'):
        if self.backbone_name == 'vit':
            vis = self.backbone(img)
            vis = vis.unsqueeze(-1).unsqueeze(-1)
        else:
            vis = self.backbone(img)
        vis   = self.vis_shared(vis)
        vqa   = self.vqa_encoder(vqa_feat)
        fused = self.fusion(
            torch.cat([vis, vqa], dim=1)
        )
        if mode == 'reg':
            return [head(fused).squeeze(-1)
                    for head in self.heads_reg]
        else:
            return [head(fused)
                    for head in self.heads_cls]

def get_predictions(outputs, mode='reg'):
    preds = []
    for output in outputs:
        if mode == 'reg':
            pred = output.round().clamp(0,2).long()
        else:
            pred = output.argmax(dim=1)
        preds.append(pred)
    return torch.stack(preds, dim=1)

def evaluate_hybrid(model, loader, mode='reg'):
    model.eval()
    all_preds  = []
    all_labels = []
    with torch.no_grad():
        for batch in loader:
            if len(batch) == 3:
                images, feat, labels = batch
                feat = feat.to(device)
            else:
                images, labels = batch
                feat = None
            images = images.to(device)
            if feat is not None:
                outputs = model(images, feat, mode)
            else:
                outputs = model(images, mode=mode)
            preds = get_predictions(outputs, mode)
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())
    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    pred_total    = all_preds.sum(axis=1)
    true_total    = all_labels.sum(axis=1)
    pred_dementia = (pred_total < CUTOFF).astype(int)
    true_dementia = (true_total < CUTOFF).astype(int)
    tn, fp, fn, tp = confusion_matrix(
        true_dementia, pred_dementia, labels=[0,1]
    ).ravel()
    sens = tp/(tp+fn) if (tp+fn) > 0 else 0
    spec = tn/(tn+fp) if (tn+fp) > 0 else 0
    try:
        auc = roc_auc_score(true_dementia, -pred_total)
    except:
        auc = 0
    return {
        'sensitivity'  : sens,
        'specificity'  : spec,
        'auc'          : auc,
        'pred_total'   : pred_total,
        'true_dementia': true_dementia,
        'all_preds'    : all_preds,
        'all_labels'   : all_labels,
    }

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])

print(f'Device       : {device}')
print(f'VQA dim      : {VQA_DIM}')
print(f'Hand dim     : {HAND_DIM}')
print(f'Combined dim : {COMBINED_DIM}')
print(f'Test set     : {len(test_df)} รูป')
print(f'VQA features : {len(vqa_features)} รูป')
print(f'Hand features: {len(hand_features)} รูป')

In [ ]:
import torch.nn as nn
import timm
from torchvision import models

from torch.utils.data import Dataset, DataLoader
import torch

class CDTCombinedDataset(Dataset):
    def __init__(self, df, image_folder,
                 vqa_feats, hand_feats,
                 aug_folder=None, transform=None):
        self.df           = df.reset_index(drop=True)
        self.image_folder = image_folder
        self.vqa_feats    = vqa_feats
        self.hand_feats   = hand_feats
        self.aug_folder   = aug_folder
        self.transform    = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        filename = row['filename']

        if (self.aug_folder and
                filename.startswith('aug')):
            img_path = f'{self.aug_folder}/{filename}'
        else:
            img_path = f'{self.image_folder}/{filename}'

        try:
            img = Image.open(img_path).convert('RGB')
        except:
            img = Image.new('RGB', (224, 224),
                           (255, 255, 255))

        if self.transform:
            img = self.transform(img)

        feat = torch.tensor(
            combined_features_to_vector(
                filename,
                self.vqa_feats,
                self.hand_feats
            ),
            dtype=torch.float32
        )

        labels = torch.tensor([
            row['domain_A'], row['domain_B'],
            row['domain_C'], row['domain_D'],
            row['domain_E']
        ], dtype=torch.float32)

        return img, feat, labels

    def forward(self, img, vqa_feat, mode='reg'):
        if self.backbone_name == 'vit':
            vis = self.backbone(img)
            vis = vis.unsqueeze(-1).unsqueeze(-1)
        else:
            vis = self.backbone(img)

        vis   = self.vis_shared(vis)
        vqa   = self.vqa_encoder(vqa_feat)
        fused = self.fusion(
            torch.cat([vis, vqa], dim=1)
        )

        if mode == 'reg':
            return [
                head(fused).squeeze(-1)
                for head in self.heads_reg
            ]
        else:
            return [
                head(fused)
                for head in self.heads_cls
            ]

test_m  = CDTHybridModel('vit', vqa_dim=COMBINED_DIM).to(device)
dummy_i = torch.randn(2, 3, 224, 224).to(device)
dummy_f = torch.randn(2, COMBINED_DIM).to(device)
out     = test_m(dummy_i, dummy_f)
print(f'Output: {[o.shape for o in out]}')

In [ ]:
MODEL_FOLDER_HYBRID = f'{BASE}/models_hybrid'
os.makedirs(MODEL_FOLDER_HYBRID, exist_ok=True)

CUTOFF      = 6
AUG_FOLDER  = f'{BASE}/images_human_aug'

train_df = pd.read_csv(
    f'{BASE}/labels/human_train.csv'
)
val_df   = pd.read_csv(
    f'{BASE}/labels/human_val.csv'
)
test_df  = pd.read_csv(
    f'{BASE}/labels/human_test.csv'
)

for d in [train_df, val_df, test_df]:
    d['dementia_risk'] = d['dementia_risk'].astype(bool)

TARGET_NORMAL   = 2000
TARGET_DEMENTIA = 1200

dementia_df    = train_df[train_df['dementia_risk']]
normal_df      = train_df[~train_df['dementia_risk']]
normal_sampled = normal_df.sample(
    n=min(TARGET_NORMAL, len(normal_df)),
    random_state=42
)

need_more = max(
    0, TARGET_DEMENTIA - len(dementia_df)
)
aug_rows  = []
generated = 0

if need_more > 0:
    from PIL import ImageEnhance
    import random

    while generated < need_more:
        for _, row in dementia_df.iterrows():
            if generated >= need_more:
                break
            try:
                src = (f'{HUMAN_FOLDER}/'
                       f'{row["filename"]}')
                img = Image.open(src).convert('RGB')

                factor = random.uniform(0.8, 1.2)
                img    = ImageEnhance.Brightness(
                    img
                ).enhance(factor)

                new_name = (
                    f'aug_h_{generated:05d}_'
                    f'{row["filename"]}'
                )
                img.save(
                    f'{AUG_FOLDER}/{new_name}',
                    'JPEG', quality=95
                )

                vqa_features[new_name] = vqa_features.get(
                    row['filename'], {}
                )

                new_row             = row.copy()
                new_row['filename'] = new_name
                aug_rows.append(new_row)
                generated += 1
            except:
                continue

    aug_df      = pd.DataFrame(aug_rows)
    dementia_df = pd.concat(
        [dementia_df, aug_df], ignore_index=True
    )

train_df_balanced = pd.concat(
    [normal_sampled, dementia_df],
    ignore_index=True
).sample(frac=1, random_state=42).reset_index(
    drop=True
)

print(f'Train: {len(train_df_balanced)} รูป')
print(f'  Normal  : {(~train_df_balanced["dementia_risk"]).sum()}')
print(f'  Dementia: {train_df_balanced["dementia_risk"].sum()}')

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(
        brightness=0.2, contrast=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])

train_ds = CDTCombinedDataset(
    train_df_balanced, HUMAN_FOLDER,
    vqa_features, hand_features,
    AUG_FOLDER, train_transform
)
val_ds = CDTCombinedDataset(
    val_df, HUMAN_FOLDER,
    vqa_features, hand_features,
    None, val_transform
)
test_ds = CDTCombinedDataset(
    test_df, HUMAN_FOLDER,
    vqa_features, hand_features,
    None, val_transform
)

train_loader = DataLoader(
    train_ds, batch_size=16,
    shuffle=True, num_workers=2
)
val_loader = DataLoader(
    val_ds, batch_size=16,
    shuffle=False, num_workers=2
)
test_loader = DataLoader(
    test_ds, batch_size=16,
    shuffle=False, num_workers=2
)

In [ ]:
import torch.optim as optim
from sklearn.metrics import (
    confusion_matrix, roc_auc_score,
    f1_score, accuracy_score,
    mean_absolute_error
)

DOMAIN_WEIGHTS = torch.tensor(
    [1.0, 1.5, 1.0, 1.0, 1.0]
).to(device)

def compute_loss(outputs, labels, mode='reg'):
    total = 0
    for i, output in enumerate(outputs):
        if mode == 'reg':
            loss = nn.MSELoss()(
                output, labels[:, i].float()
            )
        else:
            loss = nn.CrossEntropyLoss()(
                output, labels[:, i].long()
            )
        total += loss * DOMAIN_WEIGHTS[i]
    return total / DOMAIN_WEIGHTS.sum()

def get_predictions(outputs, mode='reg'):
    preds = []
    for output in outputs:
        if mode == 'reg':
            pred = output.round().clamp(0, 2).long()
        else:
            pred = output.argmax(dim=1)
        preds.append(pred)
    return torch.stack(preds, dim=1)

def evaluate_hybrid(model, loader, mode='reg'):
    model.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for images, vqa_feat, labels in loader:
            images   = images.to(device)
            vqa_feat = vqa_feat.to(device)
            outputs  = model(images, vqa_feat, mode)
            preds    = get_predictions(outputs, mode)
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    pred_total    = all_preds.sum(axis=1)
    true_total    = all_labels.sum(axis=1)
    pred_dementia = (pred_total < CUTOFF).astype(int)
    true_dementia = (true_total < CUTOFF).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        true_dementia, pred_dementia, labels=[0,1]
    ).ravel()

    sens = tp/(tp+fn) if (tp+fn) > 0 else 0
    spec = tn/(tn+fp) if (tn+fp) > 0 else 0
    try:
        auc = roc_auc_score(
            true_dementia, -pred_total
        )
    except:
        auc = 0

    return {
        'sensitivity': sens,
        'specificity': spec,
        'auc'        : auc,
        'pred_total' : pred_total,
        'true_dementia': true_dementia,
    }


# Train
model_name = 'hybrid_combined_vit_reg'
path       = f'{MODEL_FOLDER_HAND}/{model_name}_best.pth'

if os.path.exists(path):
    print(f'โหลด {model_name}')
    model = CDTHybridModel('vit', vqa_dim=COMBINED_DIM).to(device)

    model.load_state_dict(
        torch.load(path, map_location=device)
    )
    model.eval()
else:
    print(f'Train {model_name}')
    model = CDTHybridModel('vit', vqa_dim=COMBINED_DIM).to(device)

    optimizer = optim.Adam(
        filter(lambda p: p.requires_grad,
               model.parameters()),
        lr=1e-3, weight_decay=1e-4
    )
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, patience=3, factor=0.5
    )

    best_combined = -1
    best_state    = None
    no_improve    = 0
    epochs        = 20
    patience      = 10

    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for images, vqa_feat, labels in train_loader:
            images   = images.to(device)
            vqa_feat = vqa_feat.to(device)
            labels   = labels.to(device)

            optimizer.zero_grad()
            outputs = model(images, vqa_feat, 'reg')
            loss    = compute_loss(outputs, labels, 'reg')
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        val_results = evaluate_hybrid(
            model, val_loader, 'reg'
        )

        model.eval()
        val_loss = 0
        with torch.no_grad():
            for images, vqa_feat, labels in val_loader:
                images   = images.to(device)
                vqa_feat = vqa_feat.to(device)
                labels   = labels.to(device)
                outputs  = model(
                    images, vqa_feat, 'reg'
                )
                loss     = compute_loss(
                    outputs, labels, 'reg'
                )
                val_loss += loss.item()
        val_loss /= len(val_loader)
        scheduler.step(val_loss)

        combined = (
            val_results['sensitivity'] +
            val_results['specificity']
        ) / 2

        if combined > best_combined:
            best_combined = combined
            best_state    = {
                k: v.cpu().clone()
                for k, v in model.state_dict().items()
            }
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f'Early stop epoch {epoch+1}')
                break

        print(f'Epoch {epoch+1:2d}/{epochs} | '
              f'Loss: {train_loss:.4f}/{val_loss:.4f} | '
              f'Sens: {val_results["sensitivity"]:.3f} | '
              f'Spec: {val_results["specificity"]:.3f} | '
              f'AUC: {val_results["auc"]:.3f}')

    model.load_state_dict(best_state)
    torch.save(
        best_state,
        f'{MODEL_FOLDER_HAND}/{model_name}_best.pth'
    )
    print(f'บันทึก {model_name} แล้ว')

In [ ]:
def evaluate_hybrid(model, loader, mode='reg'):
    model.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for images, vqa_feat, labels in loader:
            images   = images.to(device)
            vqa_feat = vqa_feat.to(device)
            outputs  = model(images, vqa_feat, mode)
            preds    = get_predictions(outputs, mode)
            all_preds.append(preds.cpu())
            all_labels.append(labels.cpu())

    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()

    pred_total    = all_preds.sum(axis=1)
    true_total    = all_labels.sum(axis=1)
    pred_dementia = (pred_total < CUTOFF).astype(int)
    true_dementia = (true_total < CUTOFF).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        true_dementia, pred_dementia, labels=[0,1]
    ).ravel()

    sens = tp/(tp+fn) if (tp+fn) > 0 else 0
    spec = tn/(tn+fp) if (tn+fp) > 0 else 0
    try:
        auc = roc_auc_score(
            true_dementia, -pred_total
        )
    except:
        auc = 0

    return {
        'sensitivity'  : sens,
        'specificity'  : spec,
        'auc'          : auc,
        'pred_total'   : pred_total,
        'true_dementia': true_dementia,
        'all_preds'    : all_preds,
        'all_labels'   : all_labels,
    }

test_results = evaluate_hybrid(
    model, test_loader, 'reg'
)

print(f'Test Results — 07m:')
print(f'  Sensitivity: {test_results["sensitivity"]:.3f}')
print(f'  Specificity: {test_results["specificity"]:.3f}')
print(f'  AUC        : {test_results["auc"]:.3f}')

In [ ]:
results_all = {
    '07k ViT REG'    : {
        'sensitivity': 0.751,
        'specificity': 0.844,
        'auc'        : 0.882
    },
    '07l Hybrid VQA' : {
        'sensitivity': 0.760,
        'specificity': 0.850,
        'auc'        : 0.887
    },
    '07m Hybrid+Hand': {
        'sensitivity': test_results['sensitivity'],
        'specificity': test_results['specificity'],
        'auc'        : test_results['auc'],
    },
}

print('='*75)
print('FINAL COMPARISON')
print('='*75)
print(f'{"Model":<25} {"Sens":>10} '
      f'{"Spec":>10} {"AUC":>10}')
print('='*75)

best_auc = max(
    v['auc'] for v in results_all.values()
)

for name, r in results_all.items():
    mark = '✅' if r['auc'] == best_auc else ''
    print(f'{name:<25} '
          f'{r["sensitivity"]:>10.3f} '
          f'{r["specificity"]:>10.3f} '
          f'{r["auc"]:>10.3f} {mark}')

print('='*75)

with open(
    f'{RESULT_FOLDER}/07m_combined_results.json',
    'w'
) as f:
    json.dump({
        k: {
            m: float(v)
            for m, v in r.items()
            if isinstance(v, (int, float))
        }
        for k, r in results_all.items()
    }, f, indent=2)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
import torch
import torch.nn as nn
import timm
from torchvision import models
from PIL import Image

class CDTModel(nn.Module):
    def __init__(self, backbone_name='resnet50',
                 num_domains=5, num_classes=3):
        super().__init__()

        if backbone_name == 'resnet50':
            backbone = models.resnet50(
                weights=models.ResNet50_Weights.IMAGENET1K_V1
            )
            self.feature_dim = 2048
            for param in backbone.parameters():
                param.requires_grad = False
            self.backbone = nn.Sequential(
                *list(backbone.children())[:-1]
            )
        elif backbone_name == 'vit':
            self.backbone = timm.create_model(
                'vit_base_patch16_224',
                pretrained=True,
                num_classes=0
            )
            self.feature_dim = 768
            for param in self.backbone.parameters():
                param.requires_grad = False

        self.backbone_name = backbone_name
        self.shared = nn.Sequential(
            nn.Flatten(),
            nn.Linear(self.feature_dim, 512),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.heads_cls = nn.ModuleList([
            nn.Linear(512, num_classes)
            for _ in range(num_domains)
        ])
        self.heads_reg = nn.ModuleList([
            nn.Linear(512, 1)
            for _ in range(num_domains)
        ])

    def forward(self, x, mode='reg'):
        if self.backbone_name == 'resnet50':
            feat = self.backbone(x)
        else:
            feat = self.backbone(x)
            feat = feat.unsqueeze(-1).unsqueeze(-1)
        feat = self.shared(feat)
        if mode == 'reg':
            return [head(feat).squeeze(-1)
                    for head in self.heads_reg]
        else:
            return [head(feat)
                    for head in self.heads_cls]

def get_predictions(outputs, mode='reg'):
    preds = []
    for output in outputs:
        if mode == 'reg':
            pred = output.round().clamp(0, 2).long()
        else:
            pred = output.argmax(dim=1)
        preds.append(pred)
    return torch.stack(preds, dim=1)

In [ ]:
from torch.utils.data import Dataset, DataLoader

class CDTDataset(Dataset):
    def __init__(self, df, image_folder,
                 aug_folder=None, transform=None):
        self.df           = df.reset_index(drop=True)
        self.image_folder = image_folder
        self.aug_folder   = aug_folder
        self.transform    = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        filename = row['filename']

        if (self.aug_folder and
                filename.startswith('aug')):
            img_path = f'{self.aug_folder}/{filename}'
        else:
            img_path = f'{self.image_folder}/{filename}'

        try:
            img = Image.open(img_path).convert('RGB')
        except:
            img = Image.new('RGB', (224, 224),
                           (255, 255, 255))

        if self.transform:
            img = self.transform(img)

        labels = torch.tensor([
            row['domain_A'], row['domain_B'],
            row['domain_C'], row['domain_D'],
            row['domain_E']
        ], dtype=torch.float32)

        return img, labels

In [ ]:
import torch
import numpy as np
from torch.utils.data import DataLoader
from torchvision import transforms

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std =[0.229, 0.224, 0.225]
    )
])

vit_reg_path = (
    f'{BASE}/models_human_undersample/'
    f'human_us_vit_reg_best.pth'
)
vit_model = CDTModel(backbone_name='vit').to(device)
vit_model.load_state_dict(
    torch.load(vit_reg_path, map_location=device)
)
vit_model.eval()

test_ds_plain = CDTDataset(
    test_df, HUMAN_FOLDER, None, val_transform
)
test_loader_plain = DataLoader(
    test_ds_plain, batch_size=16, shuffle=False
)

vit_raw    = []
vit_labels = []
with torch.no_grad():
    for images, labels in test_loader_plain:
        images  = images.to(device)
        outputs = vit_model(images, mode='reg')
        raw     = torch.stack(
            [o.squeeze(-1) for o in outputs], dim=1
        )
        vit_raw.append(raw.cpu())
        vit_labels.append(labels.cpu())

vit_raw    = torch.cat(vit_raw).numpy()
vit_labels = torch.cat(vit_labels).numpy()

hybrid_path = (
    f'{BASE}/models_hybrid/'
    f'hybrid_vit_reg_best.pth'
)

from torch.utils.data import Dataset

class CDTHybridDataset(Dataset):
    def __init__(self, df, image_folder,
                 vqa_feats, aug_folder=None,
                 transform=None):
        self.df           = df.reset_index(drop=True)
        self.image_folder = image_folder
        self.vqa_feats    = vqa_feats
        self.aug_folder   = aug_folder
        self.transform    = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row      = self.df.iloc[idx]
        filename = row['filename']

        if (self.aug_folder and
                filename.startswith('aug')):
            img_path = f'{self.aug_folder}/{filename}'
        else:
            img_path = f'{self.image_folder}/{filename}'

        try:
            img = Image.open(img_path).convert('RGB')
        except:
            img = Image.new('RGB', (224, 224),
                           (255, 255, 255))

        if self.transform:
            img = self.transform(img)

        feat = torch.tensor(
            features_to_vector(
                self.vqa_feats.get(filename, {})
            ),
            dtype=torch.float32
        )

        labels = torch.tensor([
            row['domain_A'], row['domain_B'],
            row['domain_C'], row['domain_D'],
            row['domain_E']
        ], dtype=torch.float32)

        return img, feat, labels

VQA_DIM = len(features_to_vector({}))

hybrid_model = CDTHybridModel(
    'vit', vqa_dim=VQA_DIM
).to(device)
hybrid_model.load_state_dict(
    torch.load(hybrid_path, map_location=device)
)
hybrid_model.eval()

test_ds_hybrid = CDTHybridDataset(
    test_df, HUMAN_FOLDER,
    vqa_features, None, val_transform
)
test_loader_hybrid = DataLoader(
    test_ds_hybrid, batch_size=16, shuffle=False
)

hybrid_raw    = []
hybrid_labels = []
with torch.no_grad():
    for images, vqa_feat, labels in test_loader_hybrid:
        images   = images.to(device)
        vqa_feat = vqa_feat.to(device)
        outputs  = hybrid_model(images, vqa_feat, 'reg')
        raw      = torch.stack(
            [o.squeeze(-1) for o in outputs], dim=1
        )
        hybrid_raw.append(raw.cpu())
        hybrid_labels.append(labels.cpu())

hybrid_raw    = torch.cat(hybrid_raw).numpy()
hybrid_labels = torch.cat(hybrid_labels).numpy()

# ── 3. 07m Combined (VQA + Hand) ──
combined_path = (
    f'{BASE}/models_hybrid_hand/'
    f'hybrid_combined_vit_reg_best.pth'
)

combined_model = CDTHybridModel(
    'vit', vqa_dim=COMBINED_DIM
).to(device)
combined_model.load_state_dict(
    torch.load(combined_path, map_location=device)
)
combined_model.eval()

test_ds_combined = CDTCombinedDataset(
    test_df, HUMAN_FOLDER,
    vqa_features, hand_features,
    None, val_transform
)
test_loader_combined = DataLoader(
    test_ds_combined, batch_size=16, shuffle=False
)

combined_raw    = []
combined_labels = []
with torch.no_grad():
    for images, feat, labels in test_loader_combined:
        images = images.to(device)
        feat   = feat.to(device)
        outputs = combined_model(images, feat, 'reg')
        raw     = torch.stack(
            [o.squeeze(-1) for o in outputs], dim=1
        )
        combined_raw.append(raw.cpu())
        combined_labels.append(labels.cpu())

combined_raw    = torch.cat(combined_raw).numpy()
combined_labels = torch.cat(combined_labels).numpy()

print(f'\nshapes:')
print(f'  vit_raw     : {vit_raw.shape}')
print(f'  hybrid_raw  : {hybrid_raw.shape}')
print(f'  combined_raw: {combined_raw.shape}')

In [ ]:
domain_names = {
    0: 'A: digit_count',
    1: 'B: worst_quadrant',
    2: 'C: spatial',
    3: 'D: hands_present',
    4: 'E: hands_placement',
}

colors_true = {
    0: '#3F51B5',
    1: '#00BCD4',
    2: '#8BC34A',
}

In [ ]:
for name, raw in [
    ('vit_raw',     vit_raw),
    ('hybrid_raw',  hybrid_raw),
    ('combined_raw',combined_raw),
]:
    print(f'{name}:')
    print(f'  shape: {raw.shape}')
    print(f'  min  : {raw.min():.3f}')
    print(f'  max  : {raw.max():.3f}')
    print(f'  mean : {raw.mean():.3f}')
    print()

In [ ]:
from scipy.stats import gaussian_kde
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(
    3, 5, figsize=(25, 15)
)

models_compare = [
    ('07k ViT REG',  vit_raw,      vit_labels),
    ('07l Hybrid',   hybrid_raw,   hybrid_labels),
    ('07m Combined', combined_raw, combined_labels),
]

colors_true = {
    0: '#3F51B5',
    1: '#00BCD4',
    2: '#8BC34A',
}

for row_idx, (name, raw, labels) in enumerate(
    models_compare
):
    for d_idx, d_name in domain_names.items():
        ax     = axes[row_idx][d_idx]
        true_d = labels[:, d_idx].astype(int)
        score_d= raw[:, d_idx]

        plotted = False
        for true_val in [0, 1, 2]:
            mask   = (true_d == true_val)
            scores = score_d[mask]
            n      = mask.sum()

            if n < 2:
                continue

            try:
                kde    = gaussian_kde(
                    scores, bw_method=0.3
                )
                # ใช้ range จริงของข้อมูล
                x_min  = scores.min() - 0.2
                x_max  = scores.max() + 0.2
                x_vals = np.linspace(x_min, x_max, 300)
                y_vals = kde(x_vals)

                ax.plot(
                    x_vals, y_vals,
                    color=colors_true[true_val],
                    linewidth=2,
                    label=f'true={true_val} (n={n})'
                )
                ax.fill_between(
                    x_vals, y_vals,
                    color=colors_true[true_val],
                    alpha=0.15
                )
                plotted = True
            except Exception as e:
                print(f'KDE error {name} d{d_idx} '
                      f'true={true_val}: {e}')

        # เส้น cutoff
        for x in [0.5, 1.5]:
            ax.axvline(
                x=x, color='gray',
                linestyle='--', alpha=0.5
            )

        ax.set_title(
            f'{name}\n{d_name}',
            fontsize=8
        )
        ax.set_xlabel('Score', fontsize=7)
        ax.set_ylabel('Density', fontsize=7)
        ax.legend(fontsize=6, loc='upper left')

        if not plotted:
            ax.text(
                0.5, 0.5, 'No data',
                ha='center', va='center',
                transform=ax.transAxes
            )

plt.suptitle(
    'Distribution Comparison: 07k vs 07l vs 07m\n'
    'blue=y_true=0  cyan=y_true=1  green=y_true=2',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    f'{RESULT_FOLDER}/07m_3model_distribution.png',
    dpi=150, bbox_inches='tight'
)
plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix
import numpy as np

combined_preds  = []
combined_labels_cm = []

combined_model.eval()
with torch.no_grad():
    for images, feat, labels in test_loader_combined:
        images  = images.to(device)
        feat    = feat.to(device)
        outputs = combined_model(images, feat, 'reg')
        preds   = get_predictions(outputs, 'reg')
        combined_preds.append(preds.cpu())
        combined_labels_cm.append(labels.cpu())

combined_preds  = torch.cat(combined_preds).numpy()
combined_labels_cm = torch.cat(combined_labels_cm).numpy()

print(f'Predictions shape: {combined_preds.shape}')

fig, axes = plt.subplots(2, 5, figsize=(22, 9))

domain_names_cm = {
    0: 'A: digit_count',
    1: 'B: worst_quadrant',
    2: 'C: spatial',
    3: 'D: hands_present',
    4: 'E: hands_placement',
}

for d_idx, d_name in domain_names_cm.items():
    true_d = combined_labels_cm[:, d_idx].astype(int)
    pred_d = combined_preds[:, d_idx].astype(int)

    cm      = confusion_matrix(
        true_d, pred_d, labels=[0,1,2]
    )
    cm_norm = cm.astype(float)
    row_sum = cm.sum(axis=1, keepdims=True)
    row_sum[row_sum == 0] = 1
    cm_norm = cm_norm / row_sum

    ax_raw  = axes[0][d_idx]
    ax_norm = axes[1][d_idx]

    sns.heatmap(
        cm, annot=True, fmt='d',
        cmap='Blues', ax=ax_raw,
        xticklabels=['0','1','2'],
        yticklabels=['0','1','2'],
        cbar=False
    )
    acc = (true_d == pred_d).mean()
    ax_raw.set_title(
        f'{d_name}\nAcc={acc:.3f}',
        fontsize=9, fontweight='bold'
    )
    ax_raw.set_xlabel('Predicted')
    ax_raw.set_ylabel('True')

    sns.heatmap(
        cm_norm, annot=True, fmt='.2f',
        cmap='Blues', ax=ax_norm,
        xticklabels=['0','1','2'],
        yticklabels=['0','1','2'],
        cbar=False, vmin=0, vmax=1
    )
    ax_norm.set_title(
        f'{d_name}\n(normalized)',
        fontsize=9
    )
    ax_norm.set_xlabel('Predicted')
    ax_norm.set_ylabel('True')

plt.suptitle(
    'Per-Domain Confusion Matrix — 07m\n'
    'Hybrid ViT REG (Vision + VQA + Hand Features)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.savefig(
    f'{RESULT_FOLDER}/07m_confusion_matrix.png',
    dpi=150, bbox_inches='tight'
)
plt.show()

print('='*65)
print('Per-Domain Accuracy — 07m Combined')
print('='*65)
print(f'{"Domain":<25} {"Acc":>8} '
      f'{"Class0%":>9} {"Class1%":>9} {"Class2%":>9}')
print('-'*65)

for d_idx, d_name in domain_names_cm.items():
    true_d = combined_labels_cm[:, d_idx].astype(int)
    pred_d = combined_preds[:, d_idx].astype(int)

    acc = (true_d == pred_d).mean()

    class_acc = []
    for v in [0, 1, 2]:
        mask = (true_d == v)
        if mask.sum() == 0:
            class_acc.append(0)
        else:
            class_acc.append(
                (pred_d[mask] == v).mean() * 100
            )

    mark = '🟢' if acc >= 0.70 else \
           '🟡' if acc >= 0.55 else '🔴'

    print(f'{d_name:<25} {acc:>8.3f} '
          f'{class_acc[0]:>8.1f}% '
          f'{class_acc[1]:>8.1f}% '
          f'{class_acc[2]:>8.1f}% {mark}')

print('='*65)